In [ ]:
# %% Libraries 
import os
import sys
import pandas as pd
import plotnine as p9
from pathlib import Path
import pypalettes as pp
import matplotlib.colors as mcolors
import scanpy as sc

In [2]:
# %% Set Main Directory
MAIN_DIR_NAME = "cosmx_gray"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME

# add it to sys.path to set it as root directory
sys.path.insert(0, str(MAIN_DIR))
os.chdir(MAIN_DIR)

# %% Results Directories
PLOTS_DIR = Path("/mnt/data/project0062/cosmx_gray/results/comb/spatial_plots/test_plotting")

## Prepare the polygons file

In [3]:
# load the polygons vertices df
polygons = pd.read_parquet("/mnt/data/project0062/cosmx_gray/data/comb/polygons/comb-polygons-light.parquet")
polygons.head()

,cell_id,x_global_px,y_global_px
0,c_2_1_17,15210.0,126821.0
1,c_2_1_17,15208.0,126803.0
2,c_2_1_17,15068.0,126691.0
3,c_2_1_17,15053.0,126694.0
4,c_2_1_17,15047.0,126696.0


In [4]:
bytes_ = polygons.memory_usage(deep=True).sum()
print(f"{bytes_/1024**2:.2f} MB")

1723.86 MB


## Prepare the AnnData object

We want to only leave columsn for plotting and some metrics:
- area
- fov
- leiden_resolvi
- ct_resolvi
- novae_domains_8
- niche_names
- add more later...

In [5]:
# %% load anndata
CUR_OBJ_V = 'v10'
CUR_OBJ_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / f'comb_{CUR_OBJ_V}.h5ad'
comb = sc.read_h5ad(CUR_OBJ_PATH)


In [12]:
essential_cols = [
    'cell_id',
    'nCount_RNA',
    'nFeature_RNA',
    'Area.um2',
    'CenterX_global_px',
    'CenterY_global_px',
    'slide_name',
    'sample_name',
    'condition',
    'donor',
    'run',
    'modulator',
    'age',
    'mutation',
    'sex',
    'leiden_resolvi',
    'ct_resolvi',
    'novae_domains_8',
    'niche_names',

]

In [10]:
comb.obs.columns.to_list()

['fov',
 'Area',
 'AspectRatio',
 'Width',
 'Height',
 'Mean.PanCK',
 'Max.PanCK',
 'Mean.CD68',
 'Max.CD68',
 'Mean.Membrane',
 'Max.Membrane',
 'Mean.CD45',
 'Max.CD45',
 'Mean.DAPI',
 'Max.DAPI',
 'SplitRatioToLocal',
 'NucArea',
 'NucAspectRatio',
 'Circularity',
 'Eccentricity',
 'Perimeter',
 'Solidity',
 'assay_type',
 'version',
 'Run_Tissue_name',
 'Panel',
 'cellSegmentationSetId',
 'cellSegmentationSetName',
 'slide_ID',
 'CenterX_global_px',
 'CenterY_global_px',
 'unassignedTranscripts',
 'median_RNA',
 'RNA_quantile_0.75',
 'RNA_quantile_0.8',
 'RNA_quantile_0.85',
 'RNA_quantile_0.9',
 'RNA_quantile_0.95',
 'RNA_quantile_0.99',
 'nCount_RNA',
 'nFeature_RNA',
 'median_negprobes',
 'negprobes_quantile_0.75',
 'negprobes_quantile_0.8',
 'negprobes_quantile_0.85',
 'negprobes_quantile_0.9',
 'negprobes_quantile_0.95',
 'negprobes_quantile_0.99',
 'nCount_negprobes',
 'nFeature_negprobes',
 'median_falsecode',
 'falsecode_quantile_0.75',
 'falsecode_quantile_0.8',
 'falsecod

We make a lighter anndata object with only essential columns in the metadata

In [15]:
meta_light = comb.obs[essential_cols].copy()
meta_light

,cell_id,nCount_RNA,nFeature_RNA,Area.um2,CenterX_global_px,CenterY_global_px,slide_name,sample_name,condition,donor,run,modulator,age,mutation,sex,leiden_resolvi,ct_resolvi,novae_domains_8,niche_names
cell_id,,,,,,,,,,,,,,,,,,,
c_3_1_3,c_3_1_3,400,262,172.351396,85969.0,163444.0,SLIDE03,CTRL01,CTRL,CTRL01,RUN03,UNK,UNK,UNK,UNK,1,neutrophils,D1009,NaN
c_3_1_4,c_3_1_4,372,260,147.944714,86122.0,163426.0,SLIDE03,CTRL01,CTRL,CTRL01,RUN03,UNK,UNK,UNK,UNK,1,neutrophils,D1009,NaN
c_3_1_5,c_3_1_5,125,91,82.233303,86218.0,163445.0,SLIDE03,CTRL01,CTRL,CTRL01,RUN03,UNK,UNK,UNK,UNK,1,neutrophils,D1009,NaN
c_3_1_6,c_3_1_6,141,93,57.985763,86041.0,163448.0,SLIDE03,CTRL01,CTRL,CTRL01,RUN03,UNK,UNK,UNK,UNK,1,neutrophils,D1009,NaN
c_3_1_7,c_3_1_7,55,36,39.829043,85996.0,163380.0,SLIDE03,CTRL01,CTRL,CTRL01,RUN03,UNK,UNK,UNK,UNK,3,macrophages,D1009,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
c_4_192_1024,c_4_192_1024,519,306,160.733989,44075.0,-4100.0,SLIDE04,PWCF06,CF,PWCF06,RUN03,UNK,UNK,UNK,UNK,14,fibroblasts,D1014,vascular
c_4_192_1025,c_4_192_1025,53,43,64.915698,43651.0,-4114.0,SLIDE04,PWCF06,CF,PWCF06,RUN03,UNK,UNK,UNK,UNK,4,AT2,D1005,airway_support
c_4_192_1026,c_4_192_1026,61,36,23.119074,43850.0,-4103.0,SLIDE04,PWCF06,CF,PWCF06,RUN03,UNK,UNK,UNK,UNK,17,plasma,D1014,vascular


In [16]:
#check sizes 
bytes_ = comb.obs.memory_usage(deep=True).sum()
print(f"{bytes_/1024**2:.2f} MB")

1159.42 MB


In [17]:
#check sizes 
bytes_ = meta_light.memory_usage(deep=True).sum()
print(f"{bytes_/1024**2:.2f} MB")

218.72 MB


We need to make sure that we have a column in common in both the `comb.obs` and the `polygons`.

The polygons table has a `cell_id` column already which are the cell identifiers/barcodes. These IDs are the same as in `comb.obs` index.

I will create a `cell_id` column in `comb.obs` where I will copy the index of the metadata. 

This column in common will be used as identifiers for the `crate` object/dictionary.

In [7]:
comb.obs['cell_id'] = comb.obs_names
comb.obs['cell_id'].head()

cell_id
c_3_1_3    c_3_1_3
c_3_1_4    c_3_1_4
c_3_1_5    c_3_1_5
c_3_1_6    c_3_1_6
c_3_1_7    c_3_1_7
Name: cell_id, dtype: object

In [33]:
#check that comb.obs and polgons have the a column in common

def check_column_in_both(df1, df2, col):
    in1 = col in df1.columns
    in2 = col in df2.columns

    print(f"{col!r} in df1 ?  {in1}")
    print(f"{col!r} in df2 ?  {in2}")

    if in1 and in2:
        print(f"Column {col!r} is PRESENT in BOTH.")
        return True
    else:
        missing = []

        if not in1 : 
            missing.append('df1')
        if not in2 : 
            missing.append('df2')
        
        print(f"Column {col!r} MISSING from: {', '.join(missing)}")

        # return false just in case I add a pass check 
        return False

# example
check_column_in_both(df1=comb.obs, df2=polygons, col="cell_id")

'cell_id' in df1 ?  True
'cell_id' in df2 ?  True
Column 'cell_id' is PRESENT in BOTH.


True

Create a `crate` (simple dict)

In [34]:
crate = {
    "adata" : comb,
    "polygons" : polygons,
    "id_col" : "cell_id"
}

In [46]:
def subset_crate(crate, cells):

    id_col = crate["id_col"]
    
    adata = crate["adata"]
    sub_adata = adata[cells, :].copy()

    polygons = crate["polygons"]
    sub_polygons = polygons.loc[polygons[id_col].isin(cells)].copy()
    
    return {
        "adata": sub_adata, 
        "polygons": sub_polygons,
        "id_col": id_col
        }

In [37]:
cell_filter = comb.obs["sample_name"] == "CTRL01"
id_col = crate['id_col']
subset_cells = comb.obs.loc[cell_filter, id_col].to_list()

In [47]:
CTRL01_crate = subset_crate(crate, subset_cells)

In [ ]:
# imports 
import pypalettes as pp
import matplotlib.colors as mcolors
import plotnine as p9

# function
def plot_polygons(
    crate, 
    ann_var, 
    ann_type=["meta"], # implement funcitonality for gene later
    fig_size:tuple=(20,20),
    ):

    meta_df = crate["adata"].obs[ann_var].copy()
    poly_df = crate["polygons"]
    id_col = crate["id_col"]


    #### map ann_var to polygons ####
    ann_poly = pd.merge(poly_df, meta_df, on=id_col, how='left')

    #### Palette ####
    # Load the colormap
    cmap = pp.load_cmap("alphabet")
    # Get N discrete colors as hex
    # use leiden_scVI because it more clusters, keeps colouring consistent
    n = len(ann_poly[ann_var].unique()) 
    palette = [cmap(i / (n-1)) for i in range(n)]
    # Convert RGBA -> hex
    palette = [mcolors.to_hex(c) for c in palette]
    
    #### plot ####
    p = (
        p9.ggplot(
            ann_poly, 
            p9.aes(
                x="coords_x", 
                y="coords_y", 
                group=id_col, 
                fill=ann_var
            )
        )
        + p9.geom_polygon(color="white", size=0.1)
        + p9.coord_fixed(1)  # keep aspect ratio
        + p9.scale_fill_manual(values=palette)
        + p9.theme(
            axis_line=p9.element_blank(),
            axis_text=p9.element_blank(),
            axis_ticks=p9.element_blank(),
            axis_title=p9.element_blank(),
            panel_background=p9.element_rect(fill="black"),
            panel_grid_major=p9.element_blank(),
            panel_grid_minor=p9.element_blank(), 
            figure_size=fig_size,
        )
    )

    return p
    

test

In [58]:
p = plot_polygons(
    crate = CTRL01_crate,
    ann_var = "novae_domains_8",
    fig_size = (20,20)
)

save plot

In [59]:
for ext in ['png', 'svg']:
        p.save(
            filename= PLOTS_DIR / f'test.{ext}',
            dpi=900,
            units='in'
        )

/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /mnt/data/project0062/cosmx_gray/results/comb/spatial_plots/test_plotting/test.png
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 20 x 20 in image.
/mnt/data/project0062/.conda/envs/rsc_25.12/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /mnt/data/project0062/cosmx_gray/results/comb/spatial_plots/test_plotting/test.svg
